In [1]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc
import pypardiso

os.chdir("..")
%run .\pyflowsolver\volumeManager.py
%run .\pyflowsolver\sparseArray.py
%run .\pyflowsolver\fastLaplacian.py
%run .\pyflowsolver\darcySolver.py
os.chdir("notebooks")

In [2]:
SIZE = 5
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

volume_manager = VolumeManager(cond_vol)

image OK


In [3]:
if vol.shape[0] <= 1: # 30 for a 64 Gb RAM system
    dense_A, dense_b = volume_manager.get_linear_system()
    solution_template = np.linalg.solve(dense_A, dense_b)
    raveled_template = volume_manager.ravel_dense_solution(solution_template)
else:
    raveled_template = None

In [4]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [5]:
if sparse_b.size < 100:
    start_time = time.perf_counter()
    solution, error, iterations = solver.solve_jit(
        sparse_A, 
        sparse_b, 
        parallel=12, 
        max_iterations=100000, 
        target_error=1e-6,
    )
    run_time = time.perf_counter() - start_time
    if raveled_template:
        raveled_solution = volume_manager.ravel_sparse_solution(solution)
        diff = np.abs(raveled_solution - raveled_template)
        print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
    else:
        print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: [9.728511e-07]   Iterations: 128   Run time: 3.8545996001921594


In [6]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations, # sqrt(n) for n x n system
        target_error=1.0e-6, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 8.175095437989942e-07   Iterations: 33   Run time: 3.491658200044185


In [7]:
for i, j in ((a, b) for a in range(1, 1) for b in range(1)):
    start_time = time.perf_counter()
    max_iterations = sparse_b.size
    solution, error, iterations = solver._solve_cg(
            sparse_A.val,
            sparse_A.col_idx,
            sparse_A.row_ptr,
            sparse_b,
            max_iterations=max_iterations, # sqrt(n) for n x n system
            target_error=1.0e-5, # 1.0e-6
            X0=np.zeros(sparse_b.size, dtype=np.float64),
            threads=1,
        )
    run_time = time.perf_counter() - start_time

    print(f"Threads: {i}   Error: {error}   Iterations: {iterations}   Run time: {run_time}")
    